Imports:

In [1]:
from __future__ import annotations
from collections import defaultdict
from typing import (
    Callable,
    Dict,
    FrozenSet,
    Iterable,
    Optional,
    Sequence,
    Tuple,
    Union,
    cast,
)

import functools
import math
import numpy as np
import matplotlib.pyplot as plt
import itertools

from qiskit import QuantumCircuit
from qiskit.circuit.library import XXMinusYYGate, XXPlusYYGate
from qiskit.result import QuasiDistribution, Counts
from qiskit_nature.second_q.circuit.library import FermionicGaussianState
from qiskit_nature.second_q.hamiltonians import QuadraticHamiltonian
from qiskit_nature.second_q.operators import FermionicOp

Helper functions:

In [53]:
def orbital_combinations(
    n_modes: int, threshold: Optional[int] = None
) -> Iterable[tuple[int, ...]]:
    if threshold is None:
        threshold = n_modes
    yield ()
    yield tuple(range(n_modes))
    for i in range(threshold):
        yield (i,)
        yield tuple(range(i)) + tuple(range(i + 1, n_modes))

def orbital_permutations(n_modes: int) -> Iterable[tuple[int, ...]]:
    permutation = list(range(n_modes))
    for _ in range(math.ceil(n_modes / 2)):
        yield tuple(permutation)
        for i in range(0, n_modes - 1, 2):
            a, b = permutation[i], permutation[i + 1]
            permutation[i], permutation[i + 1] = b, a
        for i in range(1, n_modes - 1, 2):
            a, b = permutation[i], permutation[i + 1]
            permutation[i], permutation[i + 1] = b, a

def measurement_labels(n_modes: int) -> Iterable[tuple[tuple[int, ...], str]]:
    yield tuple(range(n_modes)), "number"
    for permutation in orbital_permutations(n_modes):
        yield permutation, "tunneling_plus_even"
        yield permutation, "tunneling_plus_odd"
        yield permutation, "tunneling_minus_even"
        yield permutation, "tunneling_minus_odd"
        yield permutation, "superconducting_plus_even"
        yield permutation, "superconducting_plus_odd"
        yield permutation, "superconducting_minus_even"
        yield permutation, "superconducting_minus_odd"

@functools.lru_cache
def kitaev_hamiltonian(
    n_modes: int,
    tunneling: float,
    superconducting: Union[float, complex],
    chemical_potential: float,
) -> QuadraticHamiltonian:
    eye = np.eye(n_modes)
    upper_diag = np.diag(np.ones(n_modes - 1), k=1)
    lower_diag = np.diag(np.ones(n_modes - 1), k=-1)
    hermitian_part = -tunneling * (upper_diag + lower_diag) + chemical_potential * eye
    antisymmetric_part = superconducting * (upper_diag - lower_diag)
    constant = -0.5 * chemical_potential * n_modes
    return QuadraticHamiltonian(
        hermitian_part=hermitian_part,
        antisymmetric_part=antisymmetric_part,
        constant=constant,
    )

def measure_interaction_op(circuit: QuantumCircuit, label: str) -> QuantumCircuit:
    if label == "number":
        circuit = circuit.copy()
        circuit.measure_all()
        return circuit
    if label.startswith("tunneling_plus"):
        gate = XXPlusYYGate(np.pi / 2, -np.pi / 2)
    elif label.startswith("tunneling_minus"):
        gate = XXPlusYYGate(np.pi / 2, -np.pi)
    elif label.startswith("superconducting_plus"):
        gate = XXMinusYYGate(np.pi / 2, -np.pi / 2)
    else:
        gate = XXMinusYYGate(np.pi / 2, -np.pi)
    if label.endswith("even"):
        start_index = 0
    else:
        start_index = 1
    circuit = circuit.copy()
    for i in range(start_index, circuit.num_qubits-1, 2):
        circuit.append(gate, [i, i+1])
    circuit.measure_all()
    return circuit

def _all_real_rz_gates(circuit: QuantumCircuit, rtol=1e-5, atol=1e-8) -> bool:
    for instruction in circuit.data:
        if isinstance(instruction.operation, XXPlusYYGate):
            _, beta = instruction.operation.params
            if not np.isclose(
                (beta + np.pi / 2) % np.pi, 0.0, rtol=rtol, atol=atol
            ) and not np.isclose((beta + np.pi / 2) % np.pi, np.pi, atol=1e-8):
                return False
    return True

Parameters:

In [91]:
n_modes = 7
tunneling_values = [-1.0]
superconducting_values = [1.0]
chemical_potential_values = list(np.linspace(0.0, 3.0, num=50))
occupied_orbitals_list = list(orbital_combinations(n_modes, threshold=2))

Circuit generation:

In [55]:
circuits = {}
for (
    tunneling, 
    superconducting, 
    chemical_potential, 
    occupied_orbitals
) in itertools.product(
    tunneling_values, 
    superconducting_values, 
    chemical_potential_values, 
    occupied_orbitals_list
):
    for permutation, label in measurement_labels(n_modes):
        hamiltonian = kitaev_hamiltonian(
            n_modes=n_modes,
            tunneling=tunneling,
            superconducting=superconducting,
            chemical_potential=chemical_potential,
        )
        transformation_matrix, _, _ = hamiltonian.diagonalizing_bogoliubov_transform()
        perm = np.array(permutation)
        full_permutation = np.concatenate([perm, perm+n_modes])
        for i in range(n_modes):
            transformation_matrix[i, :] = transformation_matrix[i, full_permutation]
        base_circuit = FermionicGaussianState(transformation_matrix, occupied_orbitals)
        if "_minus_" in label and _all_real_rz_gates(base_circuit, atol=1e-6):
            continue
        circuits[(tunneling, superconducting, chemical_potential, occupied_orbitals, permutation, label)] = measure_interaction_op(base_circuit, label)
len(circuits)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/operators/tensor.py:170: RuntimeWarning: divide by zero encountered in matmul
  ret = ufunc(*new_inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/operators/tensor.py:170: RuntimeWarning: overflow encountered in matmul
  ret = ufunc(*new_inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/operators/tensor.py:170: RuntimeWarning: invalid value encountered in matmul
  ret = ufunc(*new_inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/hamiltonians/quadratic_hamiltonian.py:327: RuntimeWarning: divide by zero encountered in matmul
  diagonalizing_unitary = majorana_basis.T.conj() @ basis_change @ majorana_basis
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-p

3900

Helper functions for creating ideal graphs:

In [5]:
_CovarianceDict = Dict[FrozenSet[Tuple[int, int]], float]

def expectation_from_correlation_matrix(
    operator: Union[QuadraticHamiltonian, FermionicOp],
    corr: np.ndarray,
    cov: Optional[_CovarianceDict] = None,
) -> tuple[complex, float]:
    dim, _ = corr.shape
    n = dim // 2
    if isinstance(operator, QuadraticHamiltonian):
        exp_val = (
            np.sum(
                operator.hermitian_part * corr[:n, :n]
                + np.real(operator.antisymmetric_part * corr[:n, n:])
            )
            + operator.constant
        )
        var = 0 + 0j
        if cov is not None:
            for i in range(n):
                for j in range(i + 1, n):
                    for k in range(n):
                        for ell in range(k + 1, n):
                            var += 2 * np.real(
                                operator.hermitian_part[i, j]
                                * operator.hermitian_part[k, ell]
                                * cov[frozenset([(i, j), (k, ell)])]
                            )
                            var += 2 * np.real(
                                operator.hermitian_part[i, j]
                                * operator.hermitian_part[k, ell].conjugate()
                                * cov[frozenset([(i, j), (k, ell)])]
                            )
                            var += 2 * np.real(
                                operator.antisymmetric_part[i, j]
                                * operator.antisymmetric_part[k, ell]
                                * cov[frozenset([(i, j + n), (k, ell + n)])]
                            )
                            var += 2 * np.real(
                                operator.antisymmetric_part[i, j]
                                * operator.antisymmetric_part[k, ell].conjugate()
                                * cov[frozenset([(i, j + n), (k, ell + n)])]
                            )
            for i in range(n):
                for j in range(i, n):
                    var += (1 + (i != j)) * (
                        operator.hermitian_part[i, i]
                        * operator.hermitian_part[j, j]
                        * cov[frozenset([(i, i), (j, j)])]
                    )
    else:  # isinstance(operator, FermionicOp)
        exp_val = 0.0
        for term, coeff in operator.terms():
            if not term:
                exp_val += coeff
            elif len(term) == 2:
                (action_i, i), (action_j, j) = term
                exp_val += (
                    coeff * corr[i + n * (action_i == "-"), j + n * (action_j == "+")]
                )
            else:
                raise ValueError(
                    "Operator must be quadratic in the fermionic ladder operators."
                )
        var = 0 + 0j
        if cov is not None:
            for term_ij, coeff_ij in operator.terms():
                if not term_ij:
                    continue
                (action_i, i), (action_j, j) = term_ij
                sign_ij = 1
                if i > j:
                    i, j = j, i
                    action_i, action_j = action_j, action_i
                    sign_ij *= -1
                if action_i == "-":
                    sign_ij *= -1
                for term_kl, coeff_kl in operator.terms():
                    if not term_kl:
                        continue
                    (action_k, k), (action_l, ell) = term_kl
                    sign_kl = 1
                    if k > ell:
                        k, ell = ell, k
                        action_k, action_l = action_l, action_k
                        sign_kl = -1
                    if action_k == "-":
                        sign_kl *= -1
                    var += (
                        coeff_ij
                        * coeff_kl.conjugate()
                        * sign_ij
                        * sign_kl
                        * cov[
                            frozenset(
                                [
                                    (i, j + n * (action_i == action_j)),
                                    (k, ell + n * (action_k == action_l)),
                                ]
                            )
                        ]
                    )
    return exp_val, np.sqrt(np.real(var))

def majorana_op(index: int, action: int) -> FermionicOp:
    if action == 0:
        return FermionicOp({f"-_{index}": 1.0}) + FermionicOp({f"+_{index}": 1.0})
    return -1j * (FermionicOp({f"-_{index}": 1.0}) - FermionicOp({f"+_{index}": 1.0}))

def site_correlation_op(site: int) -> FermionicOp:
    return 1j * majorana_op(0, 0) @ majorana_op(site // 2, site % 2)

def edge_correlation_op(n_modes: int) -> FermionicOp:
    return site_correlation_op(2 * n_modes - 1)

@functools.lru_cache
def diagonalizing_bogoliubov_transform(
    n_modes: int,
    tunneling: float,
    superconducting: Union[float, complex],
    chemical_potential: float,
) -> tuple[np.ndarray, np.ndarray, float]:
    return kitaev_hamiltonian(
        n_modes,
        tunneling=tunneling,
        superconducting=superconducting,
        chemical_potential=chemical_potential,
    ).diagonalizing_bogoliubov_transform()

In [6]:
def expval(quasi_dist: QuasiDistribution, operator: str) -> float:
    result = 0
    quasi_dict = quasi_dist.binary_probabilities()
    for bitstring, quasiprob in quasi_dict.items():
        result += quasiprob * evaluate_diagonal_op(operator, bitstring)
    return result

def evaluate_diagonal_op(operator: str, bitstring: str) -> int:
    prod = 1
    for op, bit in zip(operator, bitstring):
        if op in ("0", "1"):
            prod *= bit == op
        elif op == "Z":
            prod *= (-1) ** (bit == "1")
    return prod

def covariance(
    quasi_dist: QuasiDistribution, op1: str, op2: str
) -> float:
    expval1 = expval(quasi_dist, op1)
    expval2 = expval(quasi_dist, op2)
    cov = 0.0
    quasi_dict = quasi_dist.binary_probabilities()
    for bitstring, quasiprob in quasi_dict.items():
        cov += (
            quasiprob
            * (evaluate_diagonal_op(op1, bitstring) - expval1)
            * (evaluate_diagonal_op(op2, bitstring) - expval2)
        )
    return cov * -1.0 / quasi_dist.shots

def compute_interaction_matrix(
    quasis: dict[tuple[tuple[int, ...], str], QuasiDistribution],
    label: str,
) -> tuple[np.ndarray, _CovarianceDict]:
    n = len(list(quasis.keys())[0][0])
    mat = np.zeros((n, n))
    cov: _CovarianceDict = defaultdict(float)

    permutation = tuple(range(n))
    if (permutation, f"{label}_even") not in quasis and (
        permutation,
        f"{label}_odd",
    ) not in quasis:
        return mat, cov

    if label == "tunneling_plus":
        sign = -1
        symmetry = 1
    elif label == "tunneling_minus":
        sign = -1
        symmetry = -1
    elif label == "superconducting_plus":
        sign = 1
        symmetry = -1
    else:  # label == "superconducting_minus"
        sign = 1
        symmetry = -1
    for permutation in orbital_permutations(n):
        even_quasis = quasis[permutation, f"{label}_even"]
        odd_quasis = quasis[permutation, f"{label}_odd"]
        for start_index in [0, 1]:
            quasi_dist = odd_quasis if start_index else even_quasis
            for i in range(start_index, n - 1, 2):
                z0 = "I" * (n - i - 1) + "Z" + "I" * i
                z1 = "I" * (n - i - 2) + "Z" + "I" * (i + 1)
                z0_expval = expval(quasi_dist, z0)
                z1_expval = expval(quasi_dist, z1)
                val = 0.5 * (z1_expval + sign * z0_expval)
                p, q = permutation[i], permutation[i + 1]
                mat[p, q] = val
                mat[q, p] = symmetry * val
    for permutation in orbital_permutations(n):
        even_quasis = quasis[permutation, f"{label}_even"]
        odd_quasis = quasis[permutation, f"{label}_odd"]
        for start_index in [0, 1]:
            quasi_dist = odd_quasis if start_index else even_quasis
            for i in range(start_index, n - 1, 2):
                z0 = "I" * (n - i - 1) + "Z" + "I" * i
                z1 = "I" * (n - i - 2) + "Z" + "I" * (i + 1)
                p, q = permutation[i], permutation[i + 1]
                if p > q:
                    p, q = q, p
                for j in range(start_index, n - 1, 2):
                    z2 = "I" * (n - j - 1) + "Z" + "I" * j
                    z3 = "I" * (n - j - 2) + "Z" + "I" * (j + 1)
                    r, s = permutation[j], permutation[j + 1]
                    if r > s:
                        r, s = s, r
                    cov[frozenset([(p, q), (r, s)])] = 0.25 * (
                        covariance(quasi_dist, z0, z2)
                        + sign * covariance(quasi_dist, z0, z3)
                        + sign * covariance(quasi_dist, z1, z2)
                        + covariance(quasi_dist, z1, z3)
                    )
    return mat, cov

def compute_correlation_matrix(
    quasis: dict[tuple[tuple[int, ...], str], QuasiDistribution]
) -> tuple[np.ndarray, _CovarianceDict]:
    n = len(list(quasis.keys())[0][0])
    tunneling_plus, tunneling_plus_cov = compute_interaction_matrix(
        quasis, "tunneling_plus"
    )
    tunneling_minus, tunneling_minus_cov = compute_interaction_matrix(
        quasis, "tunneling_minus"
    )
    superconducting_plus, superconducting_plus_cov = compute_interaction_matrix(
        quasis, "superconducting_plus"
    )
    superconducting_minus, superconducting_minus_cov = compute_interaction_matrix(
        quasis, "superconducting_minus"
    )
    tunneling_mat = 0.5 * (tunneling_plus + 1j * tunneling_minus)
    superconducting_mat = 0.5 * (superconducting_plus + 1j * superconducting_minus)
    corr = np.block(
        [
            [tunneling_mat, superconducting_mat],
            [-superconducting_mat.conj(), np.eye(n) - tunneling_mat.T],
        ],
    )
    num_quasis = quasis[(tuple(range(n)), "number")]
    for i in range(n):
        num = "I" * (n - i - 1) + "1" + "I" * i
        exp_val = expval(num_quasis, num)
        corr[i, i] = exp_val
        corr[i + n, i + n] = 1 - exp_val
    cov: _CovarianceDict = defaultdict(float)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(n):
                for ell in range(k + 1, n):
                    cov[frozenset([(i, j), (k, ell)])] = 0.25 * (
                        tunneling_plus_cov[frozenset([(i, j), (k, ell)])]
                        + tunneling_minus_cov[frozenset([(i, j), (k, ell)])]
                    )
                    cov[frozenset([(i, j + n), (k, ell + n)])] = 0.25 * (
                        superconducting_plus_cov[frozenset([(i, j), (k, ell)])]
                        + superconducting_minus_cov[frozenset([(i, j), (k, ell)])]
                    )
    for i in range(n):
        z0 = "I" * (n - i - 1) + "Z" + "I" * i
        for j in range(i, n):
            z1 = "I" * (n - j - 1) + "Z" + "I" * j
            cov[frozenset([(i, i), (j, j)])] = 0.25 * covariance(num_quasis, z0, z1)
    return corr, cov

def counts_to_quasis(counts: dict[str, int]) -> QuasiDistribution:
    shots = sum(counts.values())
    data = {bitstring: count / shots for bitstring, count in counts.items()}
    return QuasiDistribution(data, shots=shots)

def post_select_quasis(
    quasis: QuasiDistribution, predicate: Callable[[str], bool]
) -> tuple[QuasiDistribution, float]:
    new_quasis = quasis.copy()
    removed_mass = 0.0
    for bitstring in new_quasis:
        if not predicate(str(bitstring)):
            removed_mass += new_quasis[bitstring]
            new_quasis[bitstring] = 0.0
    normalization = sum(new_quasis.values())
    for bitstring in new_quasis:
        new_quasis[bitstring] /= normalization
    return (
        QuasiDistribution(
            new_quasis,
            shots=int(quasis.shots * (1 - removed_mass)),
        ),
        removed_mass,
    )

def purify_idempotent_matrix(
    mat: np.ndarray, tol: float = 1e-8, max_iter: int = 1000
) -> np.ndarray:
    dim, _ = mat.shape
    three = 3 * np.eye(dim, dtype=mat.dtype)
    error = np.inf
    iterations = 0
    while error > tol and iterations < max_iter:
        mat = mat @ mat @ (three - 2 * mat)
        error = cast(float, np.linalg.norm(mat @ mat - mat))
        iterations += 1
    if error > tol:
        raise RuntimeError("Purification failed to converge.")
    return mat

def fidelity_witness(
    corr: np.ndarray,
    corr_target: np.ndarray,
    cov: Optional[_CovarianceDict] = None,
) -> tuple[float, float]:
    dim, _ = corr.shape
    n = dim // 2
    witness = 1 - np.trace((corr_target - corr) @ (corr_target - 0.5 * np.eye(dim)))

    var = 0.0
    if cov is not None:
        for i in range(n):
            for j in range(i + 1, n):
                for k in range(n):
                    for ell in range(k + 1, n):
                        var += 8 * np.real(
                            corr_target[i, j]
                            * corr_target[k, ell]
                            * cov[frozenset([(i, j), (k, ell)])]
                        )
                        var += 8 * np.real(
                            corr_target[i, j]
                            * corr_target[k, ell].conjugate()
                            * cov[frozenset([(i, j), (k, ell)])]
                        )
                        var += 8 * np.real(
                            corr_target[i, j + n]
                            * corr_target[k, ell + n]
                            * cov[frozenset([(i, j + n), (k, ell + n)])]
                        )
                        var += 8 * np.real(
                            corr_target[i, j + n]
                            * corr_target[k, ell + n].conjugate()
                            * cov[frozenset([(i, j + n), (k, ell + n)])]
                        )
        for i in range(n):
            for j in range(i, n):
                var += (1 + (i != j)) * (
                    (1 - 2 * corr_target[i, i])
                    * (1 - 2 * corr_target[j, j])
                    * cov[frozenset([(i, i), (j, j)])]
                )

    return np.real(witness), np.sqrt(np.real(var))

def compute_fidelity_witness(
    labels: list[str],
    corr: dict[
        tuple[
            float, Union[float, complex], float, tuple[int, ...], Optional[str], str
        ],
        tuple[np.ndarray, _CovarianceDict],
    ],
    n_modes: int,
    tunneling: float,
    superconducting: Union[float, complex],
    chemical_potential_values: Sequence[float],
    occupied_orbitals_list: Sequence[tuple[int, ...]],
) -> tuple[dict, dict]:
    data: dict[
        str,
        dict[str, dict[tuple[int, ...], list[tuple[float, float]]]],
    ] = {
       label: defaultdict(list) for label in labels
    }
    for chemical_potential in chemical_potential_values:
        (
            transformation_matrix,
            _,
            _,
        ) = diagonalizing_bogoliubov_transform(
            n_modes,
            tunneling=tunneling,
            superconducting=superconducting,
            chemical_potential=chemical_potential,
        )
        W1 = transformation_matrix[:, :n_modes]
        W2 = transformation_matrix[:, n_modes:]
        full_transformation_matrix = np.block([[W1, W2], [W2.conj(), W1.conj()]])
        for occupied_orbitals in occupied_orbitals_list:
            occupation = np.zeros(n_modes)
            occupation[list(occupied_orbitals)] = 1.0
            corr_diag = np.diag(np.concatenate([occupation, 1 - occupation]))
            corr_exact = (
                full_transformation_matrix.T.conj()
                @ corr_diag
                @ full_transformation_matrix
            )
            for label in labels:
                corr_mat, cov = corr[
                    tunneling,
                    superconducting,
                    chemical_potential,
                    occupied_orbitals,
                    label,
                ]
                fidelity_wit, stddev = fidelity_witness(corr_mat, corr_exact, cov)
                data[label][occupied_orbitals].append(
                    (fidelity_wit, stddev)
                )

    def zip_dict(d):
        return {k: tuple(np.array(a) for a in zip(*v)) for k, v in d.items()}

    data_zipped = {
        k1: {k2: zip_dict(v2) for k2, v2 in v1.items()} for k1, v1 in data.items()
    } 

    data_avg: dict[str, dict[str, tuple[np.ndarray, np.ndarray]]] = {}
    for label in labels:
        fidelity_witness_avg = np.zeros(len(chemical_potential_values))
        fidelity_witness_stddev = np.zeros(len(chemical_potential_values))
        for occupied_orbitals in occupied_orbitals_list:
            values, stddevs = data_zipped[label][occupied_orbitals]
            fidelity_witness_avg += np.array(values)
            fidelity_witness_stddev += np.array(stddevs) ** 2
        fidelity_witness_avg /= len(occupied_orbitals_list)
        fidelity_witness_stddev = np.sqrt(fidelity_witness_stddev) / len(
            occupied_orbitals_list
        )
        data_avg[label] = (
            fidelity_witness_avg,
            fidelity_witness_stddev,
        )

    return data_zipped, data_avg

In [35]:
tunneling = -1.0
superconducting = 1.0
chemical_potential_values = list(np.linspace(0.0, 3.0, num=5))

corr_matrices: dict[
    tuple[
        float, Union[float, complex], float, tuple[int, ...], Optional[str], str
    ],
    tuple[np.ndarray, _CovarianceDict],
] = {}
quasi_dists: dict[
    tuple[
        float, Union[float, complex], float, tuple[int, ...], Optional[str], str
    ],
    dict[tuple[tuple[int, ...], str], QuasiDistribution],
] = {}
ps_removed_masses = {}

for chemical_potential in chemical_potential_values:
    (
        transformation_matrix,
        _,
        _,
    ) = diagonalizing_bogoliubov_transform(
        n_modes,
        tunneling=tunneling,
        superconducting=superconducting,
        chemical_potential=chemical_potential,
    )
    W1 = transformation_matrix[:, : n_modes]
    W2 = transformation_matrix[:, n_modes :]
    full_transformation_matrix = np.block([[W1, W2], [W2.conj(), W1.conj()]])
    hamiltonian_parity = np.sign(
        np.real(np.linalg.det(full_transformation_matrix))
    )
    for occupied_orbitals in occupied_orbitals_list:
        exact_parity = (-1) ** len(occupied_orbitals) * hamiltonian_parity
        quasis_raw = (
            {}
        )  # dict[tuple[tuple[int, ...], str], QuasiDistribution]
        # to add: error mitigation scheme stored in dict named quasis_mem
        quasis_ps = (
            {}
        ) # dict[tuple[tuple[int, ...], str], QuasiDistribution]
        ps_removed_mass = {}  # dict[tuple[tuple[int, ...], str], float]
        for permutation, label in measurement_labels(n_modes):
            params = (tunneling, superconducting, chemical_potential, occupied_orbitals, permutation, label)
            if params in circuits:
                counts = Counts({})
                this_circuit = circuits[params]

                # replace at some point: ideal simulator just for testing
                from qiskit_aer import AerSimulator
                from qiskit import transpile
                simulator = AerSimulator()
                circ = transpile(this_circuit, simulator)

                job = simulator.run(circ, shots=2048)
                this_counts = job.result().get_counts(circ)

                for bitstring, count in this_counts.items():
                    if bitstring in counts:
                        counts[bitstring] += count
                    else:
                        counts[bitstring] = count
                
                quasis_raw[permutation, label] = counts_to_quasis(counts)
                new_quasis, removed_mass = post_select_quasis(
                    quasis_raw[permutation, label],
                    lambda bitstring: (-1)
                    ** sum(1 for b in bitstring if b == "1")
                    == exact_parity,
                )
                quasis_ps[permutation, label] = new_quasis
                ps_removed_mass[permutation, label] = removed_mass

        quasi_dists[
            tunneling,
            superconducting,
            chemical_potential,
            occupied_orbitals,
            "raw",
        ] = quasis_raw
        quasi_dists[
            tunneling,
            superconducting,
            chemical_potential,
            occupied_orbitals,
            "ps",
        ] = quasis_ps
        ps_removed_masses[
            tunneling,
            superconducting,
            chemical_potential,
            occupied_orbitals,
        ] = ps_removed_mass

        corr_matrices[
            tunneling,
            superconducting,
            chemical_potential,
            occupied_orbitals,
            "raw",
        ] = compute_correlation_matrix(quasis_raw)
        corr_mat_ps, cov_ps = compute_correlation_matrix(quasis_ps)
        corr_matrices[
            tunneling,
            superconducting,
            chemical_potential,
            occupied_orbitals,
            "ps",
        ] = (corr_mat_ps, cov_ps)
        corr_matrices[
            tunneling,
            superconducting,
            chemical_potential,
            occupied_orbitals,
            "pur",
        ] = (purify_idempotent_matrix(corr_mat_ps), cov_ps)

fw_zipped, fw_avg = compute_fidelity_witness(
    ["raw", "ps", "pur"], # todo: add mem for error mitigation
    corr_matrices,
    n_modes,
    tunneling,
    superconducting,
    chemical_potential_values,
    occupied_orbitals_list,
)
data["fidelity_witness"] = fw_zipped
data["fidelity_witness_avg"] = fw_avg

data["fidelity_witness"]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/operators/tensor.py:170: RuntimeWarning: divide by zero encountered in matmul
  ret = ufunc(*new_inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/operators/tensor.py:170: RuntimeWarning: overflow encountered in matmul
  ret = ufunc(*new_inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/operators/tensor.py:170: RuntimeWarning: invalid value encountered in matmul
  ret = ufunc(*new_inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/qiskit_nature/second_q/hamiltonians/quadratic_hamiltonian.py:327: RuntimeWarning: divide by zero encountered in matmul
  diagonalizing_unitary = majorana_basis.T.conj() @ basis_change @ majorana_basis
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-p

AttributeError: 'list' object has no attribute 'items'